# Reality AI — Qwen3 query-understanding LoRA (Kaggle T4)

Fine-tunes **Qwen/Qwen3-4B-Instruct-2507** (4-bit QLoRA) to turn one customer message into a JSON object of search filters:
`is_property_query, bhk, locality, price_min, price_max, property_type, keywords`. Prices come out as `{"amount", "unit"}` exactly as written; rupee conversion happens in the app's code, never in the model. Replies are templated by the app — this model never writes chat text.

Data: `train.jsonl` / `val.jsonl` from `python -m ai.finetune.prepare_extraction_dataset`, uploaded as a Kaggle Dataset. `val.jsonl` uses **held-out phrasing templates**. A high val score on generated data proves little; the hand-written `ai/nlu/eval_set.jsonl` in the repo is the real test.

> ⚠️ **GPU quota.** Settings → Accelerator → **GPU T4** (either T4 option; only one GPU is used). Run Cells 2–7 with `SMOKE_TEST = True` first (~30 steps): this is where library-version problems surface cheaply, and it prints a time estimate for the full run. Then set `SMOKE_TEST = False` and use **Save Version → Save & Run All (Commit)** for the real run, so it doesn't depend on the browser staying open.

**Cell 2 — Install.** Kaggle's preinstalled bitsandbytes is often stale; reinstall it cleanly. Qwen3 needs `transformers >= 4.51`. Then **restart the session** (Run → Restart & clear cell outputs) and continue from Cell 3.

In [ ]:
!pip uninstall -y -q bitsandbytes
!pip install -q --no-cache-dir "bitsandbytes>=0.45.3"
!pip install -q -U "transformers>=4.51" peft accelerate datasets trl

# importlib.metadata reads versions without importing torch (it must not start CUDA before the restart).
import importlib.metadata as md
for pkg in ["torch", "transformers", "peft", "trl", "bitsandbytes", "accelerate", "datasets"]:
    try:
        print(f"{pkg:<13} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<13} NOT INSTALLED")
print("\n>>> Now restart the session (Run > Restart & clear cell outputs), then run from Cell 3. <<<")

**Cell 3 — Imports, GPU check, config and helpers.** Two separate output directories: the trainer's scratch (never zipped) and the final adapter (the only thing zipped). The helpers are copies of `ai/finetune/train_lora_debug.py` and the validation logic mirrors `ai/nlu/schema.py` — the notebook can't import the repo, so keep them in sync by hand.

In [ ]:
import os
# One GPU only. With "T4 x2" the Trainer would wrap the 4-bit model in DataParallel, which breaks it.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import dataclasses, inspect, json, math, shutil, time
from pathlib import Path

import numpy as np
import torch

assert torch.cuda.is_available(), "No GPU. Settings > Accelerator > GPU T4, then restart."
print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
import transformers, peft, trl, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__,
      "| bitsandbytes", bitsandbytes.__version__)

# ---------------- config ----------------
DATASET_SLUG = "reality-ai-nlu"            # <-- your Kaggle Dataset's slug
DATA_DIR = Path(f"/kaggle/input/{DATASET_SLUG}")
TRAIN_FILE, VAL_FILE = DATA_DIR / "train.jsonl", DATA_DIR / "val.jsonl"

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
TRAINER_SCRATCH_DIR = Path("/kaggle/working/_scratch")      # never zipped
FINAL_ADAPTER_DIR = Path("/kaggle/working/qwen3-nlu-lora")  # the only thing zipped

EPOCHS = 3
LR = 2e-4
BATCH = 2
GRAD_ACCUM = 8                # effective batch size 16
SMOKE_TEST = True             # True: 30 steps + quick check, then stop. False: full run.
SMOKE_STEPS = 30
LOG_EVERY = 5
MAX_NEW_TOKENS = 160
MAX_ADAPTER_MB = 500

# Must match ai/finetune/configs/lora_config_full.yaml
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

if not TRAIN_FILE.exists():
    print("Inputs found under /kaggle/input:", [str(p) for p in Path("/kaggle/input").glob("*/*")])
    raise FileNotFoundError(f"{TRAIN_FILE} not found — set DATASET_SLUG to your dataset's folder name.")

# ---------------- helpers (copies of train_lora_debug.py) ----------------
def template_kwargs(tokenizer):
    # enable_thinking=False for templates that support it; Instruct-2507 has no thinking mode.
    return {"enable_thinking": False} if "enable_thinking" in (tokenizer.chat_template or "") else {}


def dtype_kwarg(dtype):
    # transformers >= 4.56 renamed from_pretrained(torch_dtype=) to dtype=.
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    return {"dtype": dtype} if (major, minor) >= (4, 56) else {"torch_dtype": dtype}


def tokenize_example(example, tokenizer, kwargs):
    # Loss only on the assistant's JSON: prompt tokens are masked to -100.
    messages = example["messages"]
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True, **kwargs)
    full = tokenizer.apply_chat_template(messages, tokenize=False, **kwargs)
    if not full.startswith(prompt):
        raise ValueError(f"Template renders the prompt differently with the answer appended.\n{prompt!r}\n{full!r}")
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt tokenization changes when the answer is appended.")
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids),
            "labels": [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]}


def measure_max_len(lengths):
    p95 = float(np.percentile(lengths, 95))
    return int(min(512, max(256, math.ceil(p95 / 64) * 64)))


def build_trainer(model, tokenizer, train_ds, output_dir, max_len, *, epochs, lr, batch, grad_accum,
                  bf16, max_steps=-1, callbacks=None):
    # Version-tolerant: only pass SFTConfig fields / SFTTrainer params this TRL version has.
    from transformers import DataCollatorForSeq2Seq
    from trl import SFTConfig, SFTTrainer
    wanted = dict(output_dir=output_dir, num_train_epochs=epochs, max_steps=max_steps, learning_rate=lr,
                  per_device_train_batch_size=batch, gradient_accumulation_steps=grad_accum,
                  lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=LOG_EVERY,
                  save_strategy="no", report_to="none", optim="paged_adamw_8bit",
                  gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
                  fp16=not bf16, bf16=bf16, dataset_kwargs={"skip_prepare_dataset": True},
                  remove_unused_columns=False, max_length=max_len, max_seq_length=max_len)
    fields = {f.name for f in dataclasses.fields(SFTConfig)}
    if "max_length" in fields:
        wanted.pop("max_seq_length")
    dropped = [k for k in wanted if k not in fields and k not in ("max_length", "max_seq_length")]
    if dropped:
        print("[info] SFTConfig has no field(s):", dropped)
    config = SFTConfig(**{k: v for k, v in wanted.items() if k in fields})
    kwargs = dict(model=model, args=config, train_dataset=train_ds, callbacks=callbacks or [],
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
    params = inspect.signature(SFTTrainer.__init__).parameters
    kwargs["processing_class" if "processing_class" in params else "tokenizer"] = tokenizer
    return SFTTrainer(**kwargs)


def make_callbacks():
    from transformers import TrainerCallback

    class NanStop(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kw):
            loss = (logs or {}).get("loss")
            if loss is not None and not math.isfinite(loss):
                print(f"\n[STOP] loss became {loss} at step {state.global_step}. Lower LR or check data.")
                control.should_training_stop = True

    class ProgressLog(TrainerCallback):
        def __init__(self):
            self.t0 = time.time()

        def on_log(self, args, state, control, logs=None, **kw):
            if logs and "loss" in logs:
                mins = (time.time() - self.t0) / 60
                gb = torch.cuda.max_memory_allocated() / 1024**3
                print(f"step {state.global_step:>4}/{state.max_steps}  loss {logs['loss']:.4f}  "
                      f"elapsed {mins:5.1f} min  peak GPU {gb:.1f} GB")

    return [NanStop(), ProgressLog()]


# ---------------- output validation (mirrors ai/nlu/schema.py) ----------------
KEYS = ["is_property_query", "bhk", "locality", "price_min", "price_max", "property_type", "keywords"]
UNIT = {"lakh": 1e5, "crore": 1e7, "rupee": 1.0}


def first_json(text):
    start = text.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
                continue
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        obj = json.loads(text[start:i + 1])
                        return obj if isinstance(obj, dict) else None
                    except json.JSONDecodeError:
                        break
        else:
            return None
        start = text.find("{", start + 1)
    return None


def _price(p):
    if p is None:
        return None
    if not isinstance(p, dict) or isinstance(p.get("amount"), bool) or not isinstance(p.get("amount"), (int, float)) \
            or p.get("unit") not in UNIT or p["amount"] <= 0:
        raise ValueError("bad price")
    return float(p["amount"]) * UNIT[p["unit"]]


def normalize(obj):
    # Structured comparison form, or None if the output is structurally invalid.
    try:
        if not isinstance(obj, dict) or not isinstance(obj.get("is_property_query"), bool):
            return None
        bhk, kw = obj.get("bhk"), obj.get("keywords") or []
        if bhk is not None and (isinstance(bhk, bool) or not isinstance(bhk, int)):
            return None
        if not isinstance(kw, list) or not all(isinstance(k, str) for k in kw):
            return None
        return {"is_property_query": obj["is_property_query"], "bhk": bhk, "locality": obj.get("locality"),
                "price_min": _price(obj.get("price_min")), "price_max": _price(obj.get("price_max")),
                "property_type": obj.get("property_type"), "keywords": frozenset(kw)}
    except ValueError:
        return None


def field_equal(f, a, b):
    if f.startswith("price") and a is not None and b is not None:
        return math.isclose(a, b, rel_tol=1e-6)
    return a == b


def generate_batch(model, tokenizer, rows, kwargs, batch_size=16):
    # Greedy, left-padded batched generation for evaluation.
    tokenizer.padding_side = "left"
    model.eval()
    outs = []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        texts = [tokenizer.apply_chat_template(r["messages"][:-1], tokenize=False, add_generation_prompt=True, **kwargs)
                 for r in chunk]
        enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id, use_cache=True)
        outs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = "right"
    return outs


def score(rows, outputs):
    valid, exact = 0, 0
    per_field = {f: 0 for f in KEYS}
    failures = []
    for r, out in zip(rows, outputs):
        expected = normalize(json.loads(r["messages"][-1]["content"]))
        got = normalize(first_json(out))
        if got is None:
            failures.append((r["messages"][1]["content"], out))
            continue
        valid += 1
        ok = {f: field_equal(f, expected[f], got[f]) for f in KEYS}
        for f in KEYS:
            per_field[f] += ok[f]
        exact += all(ok.values())
        if not all(ok.values()):
            failures.append((r["messages"][1]["content"], out))
    n = len(rows)
    return {"n": n, "json_valid_rate": valid / n, "exact_match": exact / n,
            "field_accuracy": {f: per_field[f] / n for f in KEYS}}, failures


def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

print("config + helpers ready")

**Cell 4 — Data.** Apply the model's own chat template (never overridden), mask the prompt out of the loss, and pick `MAX_SEQ_LENGTH` from the measured length distribution. Over-length examples are dropped rather than truncated — a cut-off JSON would teach broken output.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
TKW = template_kwargs(tokenizer)
print("chat template kwargs:", TKW or "(none)")

train_rows, val_rows = load_jsonl(TRAIN_FILE), load_jsonl(VAL_FILE)
tokenized = [tokenize_example(r, tokenizer, TKW) for r in train_rows]
lengths = [len(t["input_ids"]) for t in tokenized]
MAX_SEQ_LENGTH = measure_max_len(lengths)
kept = [t for t in tokenized if len(t["input_ids"]) <= MAX_SEQ_LENGTH]
print(f"train {len(train_rows)} | val {len(val_rows)}")
print("token length percentiles 50/90/95/99/max:",
      [int(np.percentile(lengths, q)) for q in (50, 90, 95, 99)], max(lengths))
print(f"MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}; dropped {len(tokenized) - len(kept)} over-length examples")
ex = tokenized[0]
print("loss tokens of example 0:", repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))
train_ds = Dataset.from_list(kept)

**Cell 5 — Base model in 4-bit NF4.** T4 has no bf16, so compute dtype is float16 unless bf16 is genuinely supported.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

BF16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if BF16 else torch.float16
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype),
    device_map={"": 0},
    **dtype_kwarg(compute_dtype),
)
model.config.use_cache = False
print(f"loaded in {time.time() - t0:.0f}s | compute dtype {compute_dtype} | GPU {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

**Cell 6 — LoRA.** r 16, alpha 32, all seven projections. The trainable share must be between 0 and 5% — 0 means a module name didn't match.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(**LORA))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = 100 * trainable / total
print(f"trainable {trainable:,} / {total:,} = {pct:.2f}%")
assert 0 < pct < 5, f"trainable share {pct:.2f}% outside (0, 5): check target_modules"

**Cell 7 — Train.** With `SMOKE_TEST = True`: 30 steps, then JSON validity on 20 val queries, the loss trend and a time estimate for the full run — then the notebook stops on purpose. Set `SMOKE_TEST = False` in Cell 3 and do **Save & Run All (Commit)** for the real run (a fresh session, so the 30 smoke steps don't carry over).

In [ ]:
steps_per_epoch = math.ceil(len(kept) / (BATCH * GRAD_ACCUM))
full_steps = steps_per_epoch * EPOCHS
trainer = build_trainer(model, tokenizer, train_ds, str(TRAINER_SCRATCH_DIR), MAX_SEQ_LENGTH,
                        epochs=EPOCHS, lr=LR, batch=BATCH, grad_accum=GRAD_ACCUM, bf16=BF16,
                        max_steps=SMOKE_STEPS if SMOKE_TEST else -1, callbacks=make_callbacks())

t0 = time.time()
try:
    result = trainer.train()
except torch.cuda.OutOfMemoryError:
    raise SystemExit("OOM: set BATCH = 1 and GRAD_ACCUM = 16 in Cell 3 and restart.")
train_min = (time.time() - t0) / 60
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print(f"\ntrained {trainer.state.global_step} steps in {train_min:.1f} min, final train loss {result.training_loss:.4f}")
print("loss trend:", [round(l, 3) for l in losses])

if SMOKE_TEST:
    sec_per_step = train_min * 60 / max(1, trainer.state.global_step)
    model.config.use_cache = True
    sample = val_rows[:20]
    metrics, fails = score(sample, generate_batch(model, tokenizer, sample, TKW))
    print(f"\nSMOKE: JSON valid on {metrics['json_valid_rate']:.0%} of 20 val queries "
          f"(exact match {metrics['exact_match']:.0%} — low is normal after 30 steps)")
    for q, out in fails[:5]:
        print(f"  {q!r}\n    -> {out.strip()[:200]!r}")
    print(f"\n~{sec_per_step:.1f} s/step -> full run of {full_steps} steps ≈ {full_steps * sec_per_step / 3600:.1f} h of GPU time")
    falling = len(losses) >= 2 and losses[-1] < losses[0]
    print("loss is falling:", falling)
    raise RuntimeError("SMOKE TEST complete. If the loss is falling and most outputs are JSON, set "
                       "SMOKE_TEST = False in Cell 3 and run Save Version > Save & Run All (Commit).")

**Cell 8 — Evaluate on val.jsonl** (held-out templates). Greedy generation for every val query; JSON-validity rate, per-field exact match and overall exact match. Keep these numbers for the report — but remember val is generated data; the hand-written eval set in the repo is the real test.

In [ ]:
model.config.use_cache = True
t0 = time.time()
val_outputs = generate_batch(model, tokenizer, val_rows, TKW)
val_metrics, val_failures = score(val_rows, val_outputs)
val_metrics["eval_minutes"] = round((time.time() - t0) / 60, 1)
val_metrics["train"] = {"steps": trainer.state.global_step, "minutes": round(train_min, 1),
                        "final_loss": round(result.training_loss, 4), "max_seq_length": MAX_SEQ_LENGTH,
                        "epochs": EPOCHS, "lr": LR, "effective_batch": BATCH * GRAD_ACCUM, "lora": LORA}
print(json.dumps(val_metrics, indent=2))
print(f"\n{len(val_failures)} failures; first 15:")
for q, out in val_failures[:15]:
    print(f"  {q!r}\n    -> {out.strip()[:200]!r}")

**Cell 9 — Save and zip the adapter only.** Tries safetensors first, falls back to the legacy format. Only `FINAL_ADAPTER_DIR` is zipped, and the trainer scratch is deleted so the Output tab stays small.

In [ ]:
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
try:
    model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=True)
except Exception as exc:  # noqa: BLE001
    print("safetensors save failed, retrying with safe_serialization=False:", exc)
    model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=False)
tokenizer.padding_side = "right"
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
(FINAL_ADAPTER_DIR / "val_metrics.json").write_text(json.dumps(val_metrics, indent=2))

total_mb = 0.0
for f in sorted(FINAL_ADAPTER_DIR.iterdir()):
    mb = f.stat().st_size / 1024**2
    total_mb += mb
    print(f"  {f.name:<32} {mb:8.2f} MB")
print(f"  {'TOTAL':<32} {total_mb:8.2f} MB")
assert total_mb < MAX_ADAPTER_MB, f"adapter folder is {total_mb:.0f} MB — something other than the adapter was saved"

shutil.rmtree(TRAINER_SCRATCH_DIR, ignore_errors=True)
zip_path = shutil.make_archive(str(FINAL_ADAPTER_DIR), "zip", FINAL_ADAPTER_DIR)
print(f"\nzip: {zip_path} ({Path(zip_path).stat().st_size / 1024**2:.1f} MB) — download it from the Output tab")

**Cell 10 — Put the adapter in the repo.**

1. Download `qwen3-nlu-lora.zip` from this notebook's **Output** tab.
2. Unzip so the files sit directly in `ai/finetune/adapters/qwen3-nlu-lora/` (you should see `adapter_config.json`, `adapter_model.safetensors`, the tokenizer files and `val_metrics.json` there — not a nested folder).
3. That folder is gitignored on purpose; never commit the weights.
4. `ai/inference/load_model.py` loads it from there; `python -m ai.nlu.evaluate --parser hybrid --file ai/nlu/eval_set.jsonl` measures it on the hand-written eval set.

Copy the printed `val_metrics` into your notes for the report.